In [2]:
import io
import os
import boto3
import numpy as np
from PIL import Image
from pathlib import Path
from dotenv import load_dotenv
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torch

In [3]:
load_dotenv(Path("../.env"))

s3 = boto3.client(
    "s3",
    endpoint_url=os.getenv("AISTOR_ENDPOINT"),
    aws_access_key_id=os.getenv("AISTOR_ACCESS_KEY"),
    aws_secret_access_key=os.getenv("AISTOR_SECRET_KEY")
)

In [4]:
BUCKET = "fish-classifier"

classes = [
    "Black Sea Sprat",
    "Gilt-Head Bream",
    "Hourse Mackerel",
    "Red Mullet",
    "Red Sea Bream",
    "Sea Bass",
    "Shrimp",
    "Striped Red Mullet",
    "Trout"
]

class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}

print(class_to_idx)

{'Black Sea Sprat': 0, 'Gilt-Head Bream': 1, 'Hourse Mackerel': 2, 'Red Mullet': 3, 'Red Sea Bream': 4, 'Sea Bass': 5, 'Shrimp': 6, 'Striped Red Mullet': 7, 'Trout': 8}


In [5]:
from sklearn.model_selection import train_test_split

# Get all image keys
all_keys = []
paginator = s3.get_paginator('list_objects_v2')
for page in paginator.paginate(Bucket=BUCKET, Prefix="raw/"):
    for obj in page.get('Contents', []):
        all_keys.append(obj['Key'])

print(f"Total images: {len(all_keys)}")

# Split 70/15/15
train_keys, temp_keys = train_test_split(all_keys, test_size=0.3, random_state=42)
val_keys, test_keys = train_test_split(temp_keys, test_size=0.5, random_state=42)

print(f"Train: {len(train_keys)}")
print(f"Val: {len(val_keys)}")
print(f"Test: {len(test_keys)}")

Total images: 9000
Train: 6300
Val: 1350
Test: 1350


In [6]:
class FishDataset(Dataset):
    def __init__(self, keys, class_to_idx, transform=None):
        self.keys = keys
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, idx):
        key = self.keys[idx]
        class_name = key.split('/')[1]
        label = self.class_to_idx[class_name]

        response = s3.get_object(Bucket=BUCKET, Key=key)
        img = Image.open(io.BytesIO(response['Body'].read())).convert('RGB')

        if self.transform:
            img = self.transform(img)

        return img, label

In [7]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

In [8]:
train_dataset = FishDataset(train_keys, class_to_idx, transform=train_transforms)
val_dataset = FishDataset(val_keys, class_to_idx, transform=val_transforms)
test_dataset = FishDataset(test_keys, class_to_idx, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

Train batches: 197
Val batches: 43
Test batches: 43


In [9]:
images, labels = next(iter(train_loader))
print(f"Batch image shape: {images.shape}")
print(f"Batch label shape: {labels.shape}")
print(f"Label examples: {labels[:5]}")

Batch image shape: torch.Size([32, 3, 224, 224])
Batch label shape: torch.Size([32])
Label examples: tensor([3, 8, 6, 7, 4])
